In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session
from faker import Faker
import random
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
def generate_final_hr_dataset():
    session = get_active_session()
    num_employees = 12500
    num_staff = num_employees - 6

    locales = {
        "Paris": Faker('fr_FR'),
        "Berlin": Faker('de_DE'),
        "Amsterdam": Faker('nl_NL'),
        "London": Faker('en_GB'),
        "Copenhagen": Faker('da_DK'),
        "Vienna": Faker('de_AT'),
        "Madrid": Faker('es_ES')
    }

    job_map = {
    "Engineering": ["Data Engineer", "Analytics Engineer", "Frontend Engineer", "Backend Engineer"],
    "Operations": ["Solar Installer", "Maintenance Tech", "Operations Manager", "Site Supervisor"],
    "ESG Strategy": ["Climate Risk & Transition Manager", "Structural Engineer", "Grid Specialist", "Project Engineer"],
    "Sales": ["Account Manager", "Sales Representative", "Business Development", "Sales Director"],
    "Human Resources": ["People Operations Officer", "Talent Acquisition Partner", "Compensation Analyst", "Human Resources Business Partner"],
    "Product":["Product Manager","Product Analyst","Product Engineer","Product Owner","Product Designer"]
    }

    # 2026 Market Configuration
    offices = {
        'Paris': {'p': 0.30, 'base': 62000, 'curr': 'EUR', 'rate': 1.0},
        'London': {'p': 0.20, 'base': 68000, 'curr': 'GBP', 'rate': 1.22},
        'Berlin': {'p': 0.15, 'base': 60000, 'curr': 'EUR', 'rate': 1.0},
        'Madrid': {'p': 0.12, 'base': 45000, 'curr': 'EUR', 'rate': 1.0},
        'Vienna': {'p': 0.08, 'base': 55000, 'curr': 'EUR', 'rate': 1.0},
        'Copenhagen': {'p': 0.08, 'base': 520000, 'curr': 'DKK', 'rate': 0.134},
        'Amsterdam': {'p': 0.07, 'base': 63000, 'curr': 'EUR', 'rate': 1.0}
    }
    
    emp_ids = [f'VC-{i}'for i in range(10001,10001 + num_staff)]
    selected_depts = np.random.choice(list(job_map.keys()), num_staff)
    office_choices_staff = np.random.choice(list(offices.keys()), num_staff, p=[v['p'] for v in offices.values()])

    # Generate Localized Names
    first_names = []
    last_names = []
    # Pre-initializing fakers for performance
    faker_instances = {city: Faker(locale) for city, locale in locales.items()}

    for city in office_choices_staff:
        fake = faker_instances[city]
        first_names.append(fake.first_name())
        last_names.append(fake.last_name())

    # 1. Table: Raw Employees
    df_employees = pd.DataFrame({
        'EMPLOYEE_ID': emp_ids,
        'FIRST_NAME': first_names,
        'LAST_NAME': last_names,
        'OFFICE': office_choices_staff,
        'AGE': [random.randint(18,65) for _ in range(num_staff)],
        'JOB_TITLE':[np.random.choice(job_map[d]) for d in selected_depts],
        'DEPARTMENT': selected_depts,
        'CONTRACT_TYPE': np.random.choice(['Full-time', 'Part-time'], num_staff, p=[0.85, 0.15]),
        'HIRE_DATE': [datetime(2018, 1, 1) + timedelta(days=np.random.randint(0, 2900)) for _ in range(num_staff)],
        'GENDER': np.random.choice(['F', 'M', 'Non-Binary'], num_staff, p=[0.48, 0.48, 0.04])
    })

    leader_titles = ["CEO", "CTO", "CFO", "VP of ESG", "Director of Operations", "Chief People Officer"]
    emp_ids_leaders = [f'VCL-{i}' for i in range(20001,20001 + 6)]
    
    # Fixed names for high-visibility leaders
    l_first = ["Emma", "Marc", "Elena", "Alex", "Thomas", "Sophie"]
    l_last = ["Lefebvre", "Schmidt", "Santana", "Jansen", "Weber", "Dupont"]
    
    df_leaders = pd.DataFrame({
        'EMPLOYEE_ID': emp_ids_leaders,
        'FIRST_NAME': l_first,
        'LAST_NAME': l_last,
        'JOB_TITLE': leader_titles,
        'DEPARTMENT': 'Leadership',
        'AGE': [random.randint(35,45) for _ in range(6)],
        'OFFICE': 'Paris', # Headquarters
        'CONTRACT_TYPE': 'Full-time',
        'HIRE_DATE': [datetime(2018, 1, 1) for _ in range(6)],
        'GENDER': ['F', 'M', 'F', 'Non-Binary', 'M', 'F']
    })

    # Combine
    df_employees = pd.concat([df_employees, df_leaders], ignore_index=True)

    # Table 2: RAW_Compensation
    def assign_comp_and_band(row):
        conf = offices[row['OFFICE']]
        
        # Determine Band and Multiplier based on Title/Dept
        if row['DEPARTMENT'] == 'Leadership':
            band = 'L1'
            role_mult = np.random.uniform(4.5, 6.0)
        elif any(x in row['JOB_TITLE'] for x in ['Director', 'Supervisor', 'Grid Specialist']):
            band = 'L2' # Highest Staff Band
            role_mult = np.random.uniform(1.9, 2.6)
        elif any(x in row['JOB_TITLE'] for x in ['Manager', 'Senior', 'Lead']):
            band = 'L3'
            role_mult = np.random.uniform(1.5, 1.8)
        elif any(x in row['JOB_TITLE'] for x in ['Engineer', 'Analyst', 'Partner']):
            band = 'L4'
            role_mult = np.random.uniform(1.1, 1.4)
        else:
            band = 'L5' # Lowest Staff Band
            role_mult = np.random.uniform(0.8, 1.0)
            
        # Base calculation
        local_val = conf['base'] * role_mult * np.random.uniform(0.95, 1.05)
        
        # Pro-rate for Part-time
        if row['CONTRACT_TYPE'] == 'Part-time':
            local_val *= 0.6
            
        return pd.Series([band, round(local_val, 2), round(local_val * conf['rate'], 2), conf['curr']])
        
    df_comp_cols = df_employees.apply(assign_comp_and_band, axis=1)
    df_comp = pd.DataFrame({
            'EMPLOYEE_ID': df_employees['EMPLOYEE_ID'],
            'SALARY_BAND': df_comp_cols[0],
            'BASE_SALARY_LOCAL': df_comp_cols[0],
            'BASE_SALARY_EUR': df_comp_cols[1],
            'LOCAL_CURRENCY': df_comp_cols[2],
            'BONUS_PERCENT': np.random.choice([0.05, 0.10, 0.15, 0.25], num_employees, p=[0.4, 0.3, 0.2, 0.1])
        })

# --- 3. TABLE: RAW_ESG_METRICS ---
    df_esg = pd.DataFrame({
            'EMPLOYEE_ID': df_employees['EMPLOYEE_ID'],
            'ESG_SCORE': np.random.randint(40, 100, num_employees),
            'CO2_SAVED_KG': np.random.normal(500, 150, num_employees).clip(0),
            'RENEWABLE_PROJECTS': np.random.poisson(1.5, num_employees)
        })
    # Save to Snowflake
    session.write_pandas(df_employees, "RAW_EMPLOYEES", auto_create_table=True, overwrite=True)
    session.write_pandas(df_comp, "RAW_COMPENSATION", auto_create_table=True, overwrite=True)
    session.write_pandas(df_esg, "RAW_ESG_METRICS", auto_create_table=True, overwrite=True)

    csv_emp = df_employees.to_csv(index=False)
    csv_comp = df_comp.to_csv(index=False)
    csv_esg = df_esg.to_csv(index=False)
    
    return "Phase 1 Finalized: Leadership department and Title mapping applied."

generate_final_hr_dataset()

In [ ]:
session = get_active_session()

In [ ]:
# Loading Tables
df_emp = session.table('RAW_EMPLOYEES').to_pandas()
df_comp = session.table('RAW_COMPENSATION').to_pandas()
df_esg = session.table('RAW_ESG_METRICS').to_pandas()

In [ ]:
# Merging into one master analysis dataframe
df_analysis = df_emp.merge(df_comp,  on='EMPLOYEE_ID').merge(df_esg,on='EMPLOYEE_ID')

In [ ]:
print(f'Dataset Shape:{df_analysis.shape}')
print(df_analysis.info())

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"


# Gender Pay Gap
gender_pay_stats = df_analysis.groupby(['SALARY_BAND','GENDER']).agg(
    avg_salary_eur=('BASE_SALARY_EUR','mean'),
    median_salary_eur=('BASE_SALARY_EUR','median'),
    headcount=('EMPLOYEE_ID','count')
).reset_index()

display(gender_pay_stats.head())

# Sort by Band for logicial display
band_order = ['L1','L2','L3','L4','L5']
gender_pay_stats['SALARY_BAND'] = pd.Categorical(gender_pay_stats['SALARY_BAND'], categories=band_order, ordered=True)
gender_pay_stats = gender_pay_stats.sort_values(['SALARY_BAND'])


fig_pay_gap = px.bar(
    gender_pay_stats,
    x='SALARY_BAND',
    y='avg_salary_eur',
    color='GENDER',
    barmode='group',
    title='Average Salary by Band and Gender (EUR)',
    labels={'avg_salary_eur':'Average Salary (EUR)','SALARY_BAND':'Pay Grade'},
    template = 'plotly_white',
    text_auto='.2s'
)

fig_pay_gap.show()

In [ ]:
# Departmental ESG "Impact vs Investment"
dept_impact = df_analysis.groupby('DEPARTMENT').agg(
    total_payroll_eur=('BASE_SALARY_EUR','sum'),
    total_co2_saved_kg=('CO2_SAVED_KG','sum'),
    avg_esg_score=('ESG_SCORE','mean'),
    headcount=('EMPLOYEE_ID','count')
).reset_index()

display(dept_impact.head())

# Bubble Chart
fig_impact = px.scatter(
    dept_impact,
    x='total_payroll_eur',
    y='total_co2_saved_kg',
    size='headcount',
    color='avg_esg_score',
    hover_name='DEPARTMENT',
    title='ROI of GreenTech: Payroll Spend vs. Carbon Impact',
    labels={'total_payroll_eur':'Total Payroll (EUR','total_co2_saved_kg':'CO2 Saved (KG)'},
    color_continuous_scale='Viridis'
)

fig_impact.show()

In [ ]:
# Salary Band Utilization (Compa-Ratio)
# Aggregateion
band_utilization = df_analysis.groupby('SALARY_BAND').agg(
    min_salary=('BASE_SALARY_EUR','min'),
    max_salary=('BASE_SALARY_EUR','max'),
    avg_salary=('BASE_SALARY_EUR','mean'),
    std_dev=('BASE_SALARY_EUR','std')
).reset_index()

display(band_utilization)

# Visualization
fig_spread = px.box(
    df_analysis,
    x='SALARY_BAND',
    y='BASE_SALARY_EUR',
    color='SALARY_BAND',
    category_orders={'SALARY_BAND':band_order},
    title='Salary Band Spread & Overlap Analysis',
    points='outliers',
    notched=True
)
fig_spread.show()

In [ ]:
# Regional Headcount & Contract Distribution

# Aggregation
office_mix = df_analysis.groupby(['OFFICE','CONTRACT_TYPE']).size().reset_index(name='emp_count')
office_mix['pct_of_office'] = office_mix.groupby('OFFICE')['emp_count'].transform(lambda x:(x/x.sum()) * 100)

display(office_mix)

# Visualization
fig_mix = px.bar(
    office_mix,
    x='OFFICE',
    y='emp_count',
    color='CONTRACT_TYPE',
    title='Workforce Composition by Office Location',
    labels={'emp_count':'Number of Employees'},
    barmode='stack',
    text=office_mix['pct_of_office'].apply(lambda x:f'{x:.1f}%')
)

fig_mix.show()

In [ ]:
# Performance (ESG Score) Correlation

# Aggregation
corr_data = df_analysis[['AGE','BASE_SALARY_EUR','ESG_SCORE','CO2_SAVED_KG']].corr()

display(corr_data)

# Visualization
fig_corr = px.imshow(
    corr_data,
    text_auto=True,
    aspect='auto',
    title='Correlation Heatmap: Performance vs. Demographics',
    color_continuous_scale='RdBu_r'
)

fig_corr.show()